# 01 · Data Cleaning and Merging

Loads the DMV, charging station, and Census files, cleans each one, and merges them into one row per California ZIP code.

**Authors:** Andrew Salah & Bradley Shea · MSBA 502, University of San Diego

In [1]:
# Libraries
import pandas as pd          # data loading and cleaning
import numpy as np           # math
import matplotlib.pyplot as plt   # charts
import seaborn as sns        # nicer charts

## Load raw data

In [2]:
# Raw files live in data/raw/ (paths are relative to this notebooks/ folder)
DATA = "../data/raw/"

# 1. DMV vehicle registrations by ZIP (gzipped to stay under GitHub upload limits)
dmv = pd.read_csv(DATA + "dmv_vehicle_fuel_type_count_by_zip_2025.csv.gz",
                  dtype={"ZIP Code": str})

# 2. EV charging stations (Dept. of Energy / NREL)
stations = pd.read_csv(DATA + "afdc_ev_stations_ca_2026-10-12.csv",
                       dtype={"ZIP": str}, low_memory=False)

# 3-5. Census tables (skiprows=[1] drops the second header row)
income = pd.read_csv(DATA + "census_acs5_2024_B19013_median_income.csv",
                     skiprows=[1])
housing = pd.read_csv(DATA + "census_acs5_2024_B25003_tenure.csv",
                      skiprows=[1])
population = pd.read_csv(DATA + "census_acs5_2024_B01003_population.csv",
                         skiprows=[1])

# Quick check: rows and columns in each
for name, df in [("DMV", dmv), ("Stations", stations), ("Income", income),
                 ("Housing", housing), ("Population", population)]:
    print(f"{name}: {df.shape}")

DMV: (504168, 7)
Stations: (21814, 75)
Income: (33772, 5)
Housing: (33772, 9)
Population: (33772, 5)


## Clean DMV registrations

In [3]:
# Rename columns, drop out-of-state ZIPs, keep passenger vehicles only
dmv.columns = ["date", "zip", "model_year", "fuel", "make", "duty", "vehicles"]
dmv = dmv[dmv["zip"].str.fullmatch(r"\d{5}")]   # removes "OOS"
dmv = dmv[dmv["duty"] == "Light"]

# Group 9 fuel types into simpler categories
fuel_map = {"Battery Electric": "BEV", "Plug-in Hybrid": "PHEV", "Hybrid Gasoline": "Hybrid",
            "Gasoline": "Gas", "Flex-Fuel": "Gas", "Diesel and Diesel Hybrid": "Diesel"}
dmv["fuel_group"] = dmv["fuel"].map(fuel_map).fillna("Other")

# One row per ZIP: vehicle counts by fuel type
dmv_zip = dmv.pivot_table(index="zip", columns="fuel_group", values="vehicles",
                          aggfunc="sum", fill_value=0)
dmv_zip["total_vehicles"] = dmv_zip.sum(axis=1)
dmv_zip["ev"] = dmv_zip["BEV"] + dmv_zip["PHEV"]
dmv_zip["ev_share"] = dmv_zip["ev"] / dmv_zip["total_vehicles"]
dmv_zip = dmv_zip.reset_index()
print("DMV ZIPs:", len(dmv_zip))

DMV ZIPs: 4390


## Clean charging stations

In [4]:
# Keep public, currently open stations (drop private, planned, temporarily closed)
st = stations[(stations["Access Code"] == "public") & (stations["Status Code"] == "E")].copy()
st["zip"] = st["ZIP"].str.zfill(5)   # fixes one 4-digit ZIP

# Count charging ports (plugs), not just stations
for col in ["EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count"]:
    st[col] = pd.to_numeric(st[col]).fillna(0)

st_zip = st.groupby("zip").agg(
    stations=("ID", "count"),
    l1_ports=("EV Level1 EVSE Num", "sum"),
    l2_ports=("EV Level2 EVSE Num", "sum"),
    dcfc_ports=("EV DC Fast Count", "sum")).reset_index()
st_zip["total_ports"] = st_zip["l1_ports"] + st_zip["l2_ports"] + st_zip["dcfc_ports"]
print("Public open stations:", len(st), "| ZIPs with chargers:", len(st_zip),
      "| Total ports:", int(st_zip["total_ports"].sum()))

Public open stations: 20180 | ZIPs with chargers: 1245 | Total ports: 69285


## Clean Census tables

In [5]:
# Helper: fix ZIP format, keep California only (ZIPs 900-961), rename columns
def clean_census(df, cols):
    df = df.copy()
    df["zip"] = df["NAME"].str.replace("ZCTA5 ", "", regex=False)
    df = df[df["zip"].str[:3].astype(int).between(900, 961)]
    return df[["zip"] + list(cols)].rename(columns=cols)

# Income: convert text codes to numbers ("-" = no data)
inc = clean_census(income, {"B19013_001E": "median_income"})
inc["median_income"] = inc["median_income"].replace(
    {"250,000+": "250000", "2,500-": "2500", "-": np.nan})
inc["median_income"] = pd.to_numeric(inc["median_income"])

# Home ownership rate = owner-occupied / all households
hou = clean_census(housing, {"B25003_001E": "households", "B25003_002E": "owner_occupied"})
hou["home_ownership"] = hou["owner_occupied"] / hou["households"]

pop = clean_census(population, {"B01003_001E": "population"})

census = inc.merge(hou, on="zip").merge(pop, on="zip")
print("Census CA ZIPs:", len(census), "| missing income:", census["median_income"].isna().sum())

Census CA ZIPs: 1802 | missing income: 212


## Merge and build variables

In [6]:
# Join DMV + Census on ZIP; left-join chargers so ZIPs with zero chargers stay in
df = dmv_zip.merge(census, on="zip", how="inner")
df = df.merge(st_zip, on="zip", how="left")
charger_cols = ["stations", "l1_ports", "l2_ports", "dcfc_ports", "total_ports"]
df[charger_cols] = df[charger_cols].fillna(0)

# Drop tiny ZIPs (PO boxes, businesses) and ZIPs missing income
df = df[(df["total_vehicles"] >= 500) & (df["population"] >= 1000)]
df = df.dropna(subset=["median_income", "home_ownership"])

# Key metrics
df["evs_per_port"] = df["ev"] / df["total_ports"].replace(0, np.nan)
df["ports_per_1k_people"] = df["total_ports"] / df["population"] * 1000

print("Final ZIPs:", len(df), "| ZIPs with zero chargers:", (df["total_ports"] == 0).sum())
df.describe().round(2)

Final ZIPs: 1388 | ZIPs with zero chargers: 253


,BEV,Diesel,Gas,Hybrid,Other,PHEV,total_vehicles,ev,ev_share,median_income,...,owner_occupied,home_ownership,population,stations,l1_ports,l2_ports,dcfc_ports,total_ports,evs_per_port,ports_per_1k_people
count,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,...,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1388.00,1135.00,1388.00
mean,1241.82,411.26,17188.10,1458.34,13.66,327.79,20640.97,1569.61,0.07,103538.59,...,5425.19,0.60,28139.82,14.17,0.15,34.40,13.76,48.31,88.39,2.40
std,1420.36,380.50,13662.33,1229.71,18.48,327.32,15999.00,1715.93,0.06,42877.29,...,4203.74,0.19,21780.44,26.33,1.92,67.01,21.66,76.66,204.17,6.45
min,0.00,5.00,418.00,9.00,0.00,0.00,508.00,1.00,0.00,22611.00,...,0.00,0.00,1009.00,0.00,0.00,0.00,0.00,0.00,0.07,0.00
25%,155.50,144.75,5370.50,324.50,1.00,64.75,6584.50,231.50,0.03,73182.50,...,1710.50,0.48,8156.25,1.00,0.00,2.00,0.00,4.00,15.10,0.19
50%,838.00,290.00,15059.50,1295.00,7.00,262.00,18632.00,1120.00,0.06,95159.00,...,4871.50,0.63,26352.00,6.00,0.00,11.00,4.00,23.00,35.12,0.92
75%,1791.00,538.75,25266.50,2261.25,20.00,500.50,30320.75,2334.25,0.10,123754.25,...,8018.75,0.73,41997.25,16.00,0.00,38.00,20.00,60.25,81.62,2.35
max,13092.00,2440.00,79034.00,7683.00,276.00,5759.00,101503.00,14710.00,0.58,250000.00,...,24330.00,1.00,105517.00,343.00,52.00,732.00,230.00,767.00,4563.00,101.32


## Save

In [7]:
# Save the clean ZIP-level table for the modeling notebook
df.to_csv("../data/processed/ev_clean_by_zip.csv", index=False)